# 温度校正（ticket 19）

有氧能力（参考心率下的 GAP 配速）的季节波动有多大，和气温有多大关系，要不要按气温校正。结论写在 `.scratch/running-tool/issues/19-temperature-correction.md`。

- 只读：`open_analysis_db()` 用 DuckDB 以 `READ_ONLY` 方式挂载 `running.sqlite`；最后一个单元格断言库文件的修改时间没变。
- 外部气温来自 Open-Meteo 历史天气接口。只发一个 0.1° 的格点（跑步最多的那个）和日期范围，响应缓存在数据目录的 `raw/open-meteo/`。格点不打印。
- 先套用 18 确认的清洗规则（实施在 21）：`pause_gap_s = 10`；没有 GPS 或没有海拔的跑步不算；逐秒速度 > 7 m/s 累计超过 60 s 的跑步不算。
- 输出里有个人数据：提交时 `nbstripout` 会清掉输出。

In [ ]:
import bisect
import json
import statistics
import subprocess
import urllib.parse
from dataclasses import replace
from datetime import UTC, datetime, timedelta

import polars as pl

from running.aerobic import AEROBIC_TYPES, aerobic_runs
from running.config import analysis_db_path, data_dir, load_config
from running.db import open_analysis_db

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_width_chars(200)

db_path = analysis_db_path()
mtime_before = db_path.stat().st_mtime_ns
print("库修改时间:", datetime.fromtimestamp(mtime_before / 1e9))
config = load_config()
config = replace(config, run_walk=replace(config.run_walk, pause_gap_s=10))
db = open_analysis_db()
q = db.query
types = ", ".join(f"'{t}'" for t in sorted(AEROBIC_TYPES))

## 手表温度

`records.temperature` 是手表传感器温度。看有多少秒有值。

In [ ]:
q("select count(*) n_seconds, count(temperature) n_with_temperature from a.records")

全部为空：FR245 Music 的 FIT 逐秒记录里没有温度字段。所以手表温度不能当自变量，下面只用外部气温。

## 有氧能力和清洗规则

每次跑步的有氧能力用 `aerobic_runs`，和仪表盘同一套计算。另外按活动统计 GPS、海拔、超速秒数和所在格点（格点只用来判断离常跑地点多远，不输出）。

In [ ]:
runs = aerobic_runs(db, config)
per = q(f"""
select r.activity_id,
  count(r.lat) > 0 and count(r.altitude) > 0 outdoor,
  sum((r.speed > 7)::int) <= 60 gps_ok,
  round(median(r.lat), 1) la, round(median(r.lon), 1) lo
from a.records r join a.activities a using (activity_id)
where a.activity_type in ({types}) group by 1
""")
grid = per.group_by("la", "lo").len().sort("len", descending=True)
home_la, home_lo = grid.row(0)[:2]
print("格点数:", grid.height, "| 最常跑的格点占", f"{grid['len'][0] / grid['len'].sum():.1%}")
near = ((pl.col("la") - home_la).abs() <= 0.21) & ((pl.col("lo") - home_lo).abs() <= 0.21)
per = per.with_columns(near.fill_null(False).alias("near")).drop("la", "lo")
acts = q(
    "select activity_id, start_time_utc, elapsed_duration_s from a.activities"
    f" where activity_type in ({types})"
)
runs = runs.join(per, on="activity_id", how="left").join(acts, on="activity_id", how="left")
runs = runs.with_columns(pl.col("outdoor", "gps_ok", "near").fill_null(False))
runs.select(
    pl.len().alias("road_runs"),
    pl.col("valid").sum().alias("valid"),
    (pl.col("valid") & pl.col("outdoor") & pl.col("gps_ok")).sum().alias("valid_after_18"),
    pl.col("near").sum().alias("near_home"),
)

## Open-Meteo 逐小时气温

只请求最常跑的格点，覆盖它周围 ±2 格（约 ±20 km）内的跑步；更远的跑步没有气温。已有缓存就不再请求。用系统的 `curl` 下载（会验证证书）；本机 Python 的证书链验证这个站点会失败。

In [ ]:
start = runs["start_time_utc"].min()[:10]
end = (datetime.now(UTC).date() - timedelta(days=2)).isoformat()
cache = data_dir() / "raw" / "open-meteo"
cache.mkdir(parents=True, exist_ok=True)
path = next(iter(sorted(cache.glob(f"hourly_temperature_2m_{start}_*.json"))), None)
if path is None:
    path = cache / f"hourly_temperature_2m_{start}_{end}.json"
    params = urllib.parse.urlencode(
        {
            "latitude": f"{home_la:.1f}",
            "longitude": f"{home_lo:.1f}",
            "start_date": start,
            "end_date": end,
            "hourly": "temperature_2m",
            "timezone": "GMT",
        }
    )
    url = f"https://archive-api.open-meteo.com/v1/archive?{params}"
    subprocess.run(["curl", "-sSf", "--max-time", "120", "-o", str(path), url], check=True)
    print("已下载:", path.name)
else:
    print("用缓存:", path.name)
data = json.loads(path.read_text())
hourly = (
    pl.DataFrame({"t": data["hourly"]["time"], "air_c": data["hourly"]["temperature_2m"]})
    .with_columns(pl.col("t").str.to_datetime("%Y-%m-%dT%H:%M", time_unit="us"))
    .drop_nulls()
    .sort("t")
)
print(hourly.height, "小时,", hourly["t"].min(), "..", hourly["t"].max())

每次跑步取中点时刻（开始 + 总时长的一半），在前后两个整点之间线性插值。

In [ ]:
runs = runs.with_columns(
    (
        pl.col("start_time_utc").str.to_datetime("%Y-%m-%dT%H:%M:%SZ", time_unit="us")
        + pl.duration(seconds=pl.col("elapsed_duration_s").fill_null(0) / 2)
    ).alias("mid_utc")
).sort("mid_utc")
runs = runs.join_asof(
    hourly.rename({"t": "t0", "air_c": "a0"}), left_on="mid_utc", right_on="t0", strategy="backward"
)
runs = runs.join_asof(
    hourly.rename({"t": "t1", "air_c": "a1"}), left_on="mid_utc", right_on="t1", strategy="forward"
)
frac = (pl.col("mid_utc") - pl.col("t0")).dt.total_seconds() / 3600
interp = (
    pl.when(pl.col("t1") == pl.col("t0"))
    .then(pl.col("a0"))
    .otherwise(pl.col("a0") + (pl.col("a1") - pl.col("a0")) * frac)
)
ok = pl.col("near") & (pl.col("t1") - pl.col("t0") <= pl.duration(hours=1))
runs = runs.with_columns(pl.when(ok).then(interp).alias("air_c")).drop("t0", "a0", "t1", "a1")

v = (
    runs.filter(
        pl.col("valid") & pl.col("outdoor") & pl.col("gps_ok") & pl.col("air_c").is_not_null()
    )
    .with_columns(
        (1000 / pl.col("speed_mps")).alias("pace"),
        pl.col("start_time_local").str.slice(0, 10).str.to_date().alias("d"),
    )
    .sort("d")
)
print("有效、清洗后、有气温的跑步:", v.height)
print(
    "跑步时气温 5/25/50/75/95 分位:",
    [round(v["air_c"].quantile(p), 1) for p in (0.05, 0.25, 0.5, 0.75, 0.95)],
)

## 气温和有氧能力

配速（s/km）对气温回归。体能在 5 年里有起伏，所以也看去掉趋势后的残差：减去前后各半年（365 天居中窗口）的配速中位数。

In [ ]:
def centered_median(dates, values, days):
    half = timedelta(days=days // 2)
    out = []
    for d in dates:
        lo = bisect.bisect_left(dates, d - half)
        hi = bisect.bisect_right(dates, d + half)
        out.append(statistics.median(values[lo:hi]))
    return out


def fit(df, y, x):
    s = df.select(
        (pl.cov(x, y, ddof=0) / pl.col(x).var(ddof=0)).alias("slope"),
        (pl.corr(x, y) ** 2).alias("r2"),
        pl.len().alias("n"),
    ).row(0, named=True)
    return {"y": y, "x": x, **s}


v = v.with_columns(pl.Series("trend", centered_median(v["d"].to_list(), v["pace"].to_list(), 365)))
v = v.with_columns(
    (pl.col("pace") - pl.col("trend")).alias("resid"),
    pl.col("d").dt.month().alias("month"),
    ((pl.col("d").dt.month() % 12) // 3).alias(
        "season"
    ),  # 0 夏 12-2，1 秋 3-5，2 冬 6-8，3 春 9-11
)
pl.DataFrame([fit(v, "pace", "air_c"), fit(v, "resid", "air_c"), fit(v, "mean_hr", "air_c")])

按 3 °C 分档看残差：

In [ ]:
v.with_columns((pl.col("air_c") // 3 * 3).alias("air_band")).group_by("air_band").agg(
    pl.len().alias("n"), pl.col("resid").median().round(1).alias("resid_median")
).sort("air_band")

## 季节波动

按月份和季节（南半球）看残差。季节的残差按年列开（年从 7 月算起），看规律是不是每年都有。

In [ ]:
v.group_by("month").agg(
    pl.len().alias("n"),
    pl.col("air_c").median().round(1).alias("air_median"),
    pl.col("resid").median().round(1).alias("resid_median"),
).sort("month")

In [ ]:
from_july = pl.col("d").dt.year() - (pl.col("d").dt.month() < 7).cast(pl.Int32)
v.with_columns(from_july.alias("from_july")).pivot(
    on="season", index="from_july", values="resid", aggregate_function="median", sort_columns=True
).sort("from_july").with_columns(pl.exclude("from_july").round(1))

同一个月内的气温效应：减去各月平均后再对气温回归，把季节里其他因素（训练周期等）的影响扣掉。

In [ ]:
w = v.with_columns(
    (pl.col("resid") - pl.col("resid").mean().over("month")).alias("resid_in_month"),
    (pl.col("air_c") - pl.col("air_c").mean().over("month")).alias("air_in_month"),
)
total = ((v["resid"] - v["resid"].mean()) ** 2).sum()
print("月份解释的残差方差:", round(1 - (w["resid_in_month"] ** 2).sum() / total, 3))
in_month = fit(w, "resid_in_month", "air_in_month")
se = abs(in_month["slope"]) * ((1 - in_month["r2"]) / (in_month["n"] - 2) / in_month["r2"]) ** 0.5
print(f"月内斜率 {in_month['slope']:.2f} ± {1.96 * se:.2f} s/km/°C (95%)，R² {in_month['r2']:.3f}")

## 校正和标注的效果

- 回归校正：配速 − k ×（气温 − 17 °C），17 °C 约是跑步时气温的中位数。比较 k = 0、全样本斜率、月内斜率三种情况下，周序列（每周至少 2 次，取中位数）和 4 周滚动中位数的周间跳动，以及季节残差（夏、秋、冬、春）。
- 标注高温周：每周气温中位数 ≥ 22 °C（约 95 分位）或 ≥ 20 °C 的周数。

In [ ]:
rows = []
for name, k in [
    ("不校正", 0.0),
    ("全样本斜率", fit(v, "pace", "air_c")["slope"]),
    ("月内斜率", in_month["slope"]),
]:
    x = v.with_columns((pl.col("pace") - k * (pl.col("air_c") - 17)).alias("pc"))
    x = x.with_columns(
        (pl.col("pc") - pl.Series(centered_median(x["d"].to_list(), x["pc"].to_list(), 365))).alias(
            "rc"
        )
    )
    weeks = (
        x.group_by(pl.col("d").dt.truncate("1w").alias("week"))
        .agg(pl.len().alias("n"), pl.col("pc").median().alias("p"))
        .filter(pl.col("n") >= 2)
        .sort("week")
        .with_columns(pl.col("p").rolling_median(4).alias("roll"))
    )
    seasons = x.group_by("season").agg(pl.col("rc").median()).sort("season")["rc"].to_list()
    rows.append(
        {
            "method": name,
            "k": round(k, 2),
            "weeks": weeks.height,
            "weekly_jump_sd": round(weeks["p"].diff().std(), 1),
            "rolling_jump_sd": round(weeks["roll"].diff().std(), 1),
            "season_resid": [round(a, 1) for a in seasons],
        }
    )
pl.DataFrame(rows)

In [ ]:
v.group_by(pl.col("d").dt.truncate("1w").alias("week")).agg(
    pl.len().alias("n"),
    pl.col("air_c").median().alias("air"),
    pl.col("resid").median().alias("resid"),
).filter(pl.col("n") >= 2).select(
    pl.len().alias("weeks"),
    (pl.col("air") >= 20).sum().alias("weeks_ge_20"),
    (pl.col("air") >= 22).sum().alias("weeks_ge_22"),
    pl.col("resid").filter(pl.col("air") >= 22).median().round(1).alias("resid_ge_22"),
    pl.col("resid").filter(pl.col("air") < 22).median().round(1).alias("resid_lt_22"),
)

In [ ]:
db.close()
mtime_after = db_path.stat().st_mtime_ns
print("库修改时间:", datetime.fromtimestamp(mtime_after / 1e9))
assert mtime_after == mtime_before, "running.sqlite 被改动了"